# Notebook 01 — Data Preprocessing

**Research topic:** *A Machine Learning Approach to Predicting Retail Investor Dormancy on Digital Investment Platforms*

### Purpose
This notebook prepares the raw customer and transaction datasets for the later feature-engineering and modelling stages.

The preprocessing stage focuses on:
- loading the two datasets,
- inspecting their structure and data quality,
- converting dates to the correct format,
- removing records that cannot be used,
- checking duplicates,
- checking missing values,
- separating the transaction data into observation and prediction periods and
- saving clean datasets for the feature-engineering notebook.

**Important:** The dormancy target is created in the feature-engineering notebook because it depends on customer-level transaction behaviour across two time periods.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

DATA_DIR = Path(".")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

TRANSACTIONS_FILE = DATA_DIR / "transactions_data.csv"
CUSTOMERS_FILE = DATA_DIR / "customer_data.csv"

CUTOFF_DATE = pd.Timestamp("2023-09-30")
PREDICTION_END = pd.Timestamp("2023-12-29")


In [2]:
# Loading the raw datasets

transactions = pd.read_csv(TRANSACTIONS_FILE)
customers = pd.read_csv(CUSTOMERS_FILE)

print("Transactions shape:", transactions.shape)
print("Customer shape:", customers.shape)

print("\nTransaction columns:")
print(transactions.columns.tolist())

print("\nCustomer columns:")
print(customers.columns.tolist())


Transactions shape: (3159157, 4)
Customer shape: (48723, 54)

Transaction columns:
['customer_id', 'date', 'amount', 'type']

Customer columns:
['customer_id', 'age', 'gender', 'location', 'income_bracket', 'occupation', 'education_level', 'marital_status', 'household_size', 'acquisition_channel', 'customer_segment', 'savings_account', 'credit_card', 'personal_loan', 'investment_account', 'insurance_product', 'active_products', 'app_logins_frequency', 'feature_usage_diversity', 'bill_payment_user', 'auto_savings_enabled', 'credit_utilization_ratio', 'international_transactions', 'failed_transactions', 'tx_count', 'avg_tx_value', 'total_tx_volume', 'first_tx', 'last_tx', 'base_satisfaction', 'tx_satisfaction', 'product_satisfaction', 'satisfaction_score', 'nps_score', 'last_survey_date', 'support_tickets_count', 'resolved_tickets_ratio', 'app_store_rating', 'feedback_sentiment', 'feature_requests', 'complaint_topics', 'clv_segment', 'monthly_transaction_count', 'average_transaction_valu

In [3]:
# Inspecting the first records

display(transactions.head())
display(customers.head())


,customer_id,date,amount,type
0,93716481,2023-11-23,1262800,Transfer
1,93716481,2023-01-05,928900,Withdrawal
2,93716481,2023-11-05,1683100,Payment
3,93716481,2023-04-14,4507350,Payment
4,93716481,2023-08-04,2096150,Payment


,customer_id,age,gender,location,income_bracket,occupation,education_level,marital_status,household_size,acquisition_channel,customer_segment,savings_account,credit_card,personal_loan,investment_account,insurance_product,active_products,app_logins_frequency,feature_usage_diversity,bill_payment_user,auto_savings_enabled,credit_utilization_ratio,international_transactions,failed_transactions,tx_count,avg_tx_value,total_tx_volume,first_tx,last_tx,base_satisfaction,tx_satisfaction,product_satisfaction,satisfaction_score,nps_score,last_survey_date,support_tickets_count,resolved_tickets_ratio,app_store_rating,feedback_sentiment,feature_requests,complaint_topics,clv_segment,monthly_transaction_count,average_transaction_value,total_transaction_volume,transaction_frequency,last_transaction_date,preferred_transaction_type,first_transaction_date,weekend_transaction_ratio,avg_daily_transactions,customer_tenure,churn_probability,customer_lifetime_value
0,93716481,44,Male,"Pereira, Risaralda",Very High,Engineer,Master,Married,5,Partnership,occasional,True,True,False,False,True,1,15,3,False,False,0.178710,1,0,76,1.679872e+06,127670300,2023-01-05,2023-12-25,9.213372,0.380,0.6,5,-7,2023-12-16,1,0.0,3.5,Neutral,Budgeting tools,NaN,Gold,1.428571,6.054025e+06,60540250,0.030769,2023-12-05,Transfer,2023-01-05,0.300000,0.030769,11.633333,0.362269,3.124148e+08
1,49020929,44,Male,"Cali, Valle del Cauca",Medium,Construction Worker,Bachelor,Married,2,Paid Ad,regular,True,False,False,False,False,5,30,0,True,False,NaN,1,0,10,5.445810e+06,54458100,2023-02-05,2023-12-20,6.073093,0.050,0.2,3,-46,2023-05-05,1,1.0,4.0,Positive,NaN,NaN,Bronze,1.571429,2.127414e+06,23401550,0.032934,2023-12-18,Transfer,2023-02-05,0.181818,0.032934,11.500000,0.181790,1.670223e+08
2,52690950,45,Male,"Barranquilla, Atlántico",High,Construction Worker,Bachelor,Married,4,Paid Ad,inactive,True,False,False,True,True,3,4,4,True,False,NaN,1,0,10,6.403350e+05,6403350,2023-01-31,2023-12-12,8.198643,0.050,0.6,4,-29,2023-11-16,0,0.0,5.0,Positive,NaN,NaN,Gold,2.500000,3.575915e+06,35759150,0.050000,2023-10-27,Transfer,2023-01-31,0.300000,0.050000,8.766667,0.321265,1.940371e+07
3,23199751,45,Male,"Bogotá, Cundinamarca",High,Electrician,High School,Single,2,Referral,inactive,True,False,False,True,False,2,15,4,True,False,NaN,1,1,23,6.601804e+05,15184150,2023-01-10,2023-12-24,7.529577,0.115,0.4,4,-31,2023-05-07,2,0.5,4.5,Positive,"Cryptocurrency trading, Budgeting tools",NaN,Bronze,1.555556,1.513354e+06,21186950,0.047782,2023-11-30,Transfer,2023-01-10,0.142857,0.047782,10.733333,0.336629,3.597811e+07
4,61997066,44,Female,"Bogotá, Cundinamarca",Medium,Cleaner,Bachelor,Married,3,Paid Ad,power,True,True,False,False,False,1,19,1,False,False,0.403606,0,0,18,1.457447e+07,262340450,2023-01-05,2023-11-27,6.605627,0.090,0.4,3,-47,2023-09-15,2,1.0,4.0,Positive,International transfers,App performance,Gold,2.111111,6.126737e+05,11640800,0.061889,2023-11-17,Payment,2023-01-05,0.315789,0.061889,11.633333,0.398151,3.046620e+08


In [4]:
# Inspecting data types and basic structure

print("TRANSACTIONS INFO")
transactions.info()

print("\nCUSTOMER INFO")
customers.info()


TRANSACTIONS INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3159157 entries, 0 to 3159156
Data columns (total 4 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   customer_id  int64 
 1   date         object
 2   amount       int64 
 3   type         object
dtypes: int64(2), object(2)
memory usage: 96.4+ MB

CUSTOMER INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48723 entries, 0 to 48722
Data columns (total 54 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   customer_id                 48723 non-null  int64  
 1   age                         48723 non-null  int64  
 2   gender                      48723 non-null  object 
 3   location                    48723 non-null  object 
 4   income_bracket              48723 non-null  object 
 5   occupation                  48723 non-null  object 
 6   education_level             48723 non-null  object 
 7   marital_status             

In [5]:
# Checking missing values

print("Missing values in transactions:")
display(transactions.isna().sum().sort_values(ascending=False))

print("\nMissing values in customer data:")
display(customers.isna().sum().sort_values(ascending=False))


Missing values in transactions:


customer_id    0
date           0
amount         0
type           0
dtype: int64


Missing values in customer data:


complaint_topics              24346
credit_utilization_ratio      18263
feature_requests              16115
location                          0
gender                            0
customer_id                       0
occupation                        0
education_level                   0
household_size                    0
marital_status                    0
customer_segment                  0
savings_account                   0
credit_card                       0
acquisition_channel               0
personal_loan                     0
investment_account                0
active_products                   0
insurance_product                 0
app_logins_frequency              0
feature_usage_diversity           0
income_bracket                    0
age                               0
auto_savings_enabled              0
bill_payment_user                 0
international_transactions        0
failed_transactions               0
total_tx_volume                   0
first_tx                    

In [6]:
# Checking for duplicated records

print("Duplicate transaction rows:", transactions.duplicated().sum())
print("Duplicate customer rows:", customers.duplicated().sum())

# Check whether customer_id is unique in the customer dataset

print( "Duplicate customer IDs:", customers["customer_id"].duplicated().sum())


Duplicate transaction rows: 102
Duplicate customer rows: 0
Duplicate customer IDs: 0


In [7]:
# Converting transaction dates to datetime
transactions["date"] = pd.to_datetime(transactions["date"], errors="coerce")

# Checking dates that could not be converted
print("Invalid/missing transaction dates:", transactions["date"].isna().sum())

# Removing records that cannot be used for temporal analysis
transactions = transactions.dropna(subset=["customer_id", "date"]).copy()

# Removing exact duplicate transaction records
transactions = transactions.drop_duplicates().copy()

print("Clean transaction shape:", transactions.shape)


Invalid/missing transaction dates: 0
Clean transaction shape: (3159055, 4)


In [13]:
# Inspect numerical transaction variables for impossible/missing values
display(transactions.describe(include="all").T)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
customer_id,3159055.0,NaN,NaN,NaN,54589666.83598,10000617.0,32120213.0,54928756.0,77940083.0,99999798.0,26413345.072923
date,3159055,NaN,NaN,NaN,2023-07-02 15:27:25.362806272,2023-01-04 00:00:00,2023-04-04 00:00:00,2023-07-03 00:00:00,2023-10-01 00:00:00,2023-12-29 00:00:00,NaN
amount,3159055.0,NaN,NaN,NaN,3971025.680749,36400.0,828600.0,1788350.0,4184800.0,689739900.0,6978593.599233
type,3159055,4,Transfer,1433908,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
# Creating clean observation and prediction-period datasets.
# No future information is used as a predictor.

observation_transactions = transactions[transactions["date"] <= CUTOFF_DATE].copy()

prediction_transactions = transactions[(transactions["date"] > CUTOFF_DATE)
    & (transactions["date"] <= PREDICTION_END)].copy()

print("Observation period:", observation_transactions["date"].min(),
      "to", observation_transactions["date"].max())

print("Prediction period:", prediction_transactions["date"].min(),
      "to", prediction_transactions["date"].max())

print("\nObservation transactions:", len(observation_transactions))
print("Prediction-period transactions:", len(prediction_transactions))


Observation period: 2023-01-04 00:00:00 to 2023-09-30 00:00:00
Prediction period: 2023-10-01 00:00:00 to 2023-12-29 00:00:00

Observation transactions: 2367794
Prediction-period transactions: 791261


In [11]:
# Saving the clean datasets for feature engineering.
observation_file = OUTPUT_DIR / "observation_transactions.csv"
prediction_file = OUTPUT_DIR / "prediction_transactions.csv"
customers_file = OUTPUT_DIR / "clean_customer_data.csv"

observation_transactions.to_csv(observation_file, index=False)

prediction_transactions.to_csv(prediction_file, index=False)

customers.to_csv(customers_file, index=False)

print("Saved:")
print(observation_file)
print(prediction_file)
print(customers_file)


Saved:
outputs\observation_transactions.csv
outputs\prediction_transactions.csv
outputs\clean_customer_data.csv


## Preprocessing outcome

The output of this notebook consists of three clean files:

- `observation_transactions.csv` — transaction activity available before the prediction point;
- `prediction_transactions.csv` — later transactions used to establish the future dormancy outcome; and
- `clean_customer_data.csv` — cleaned customer-level information.

The next notebook uses these files to create customer-level behavioural features and the binary `dormant` target.
